In [1]:
import time
import math
import numpy as np
import matplotlib.pyplot as plt
from edu1sces.model import SU2HeisenbergModel
from edu1sces.solver import solve_su2, SolverParameters
import edu1sces.core as core

In [2]:
N = 20
total_s = 0  # シングレット状態
model = SU2HeisenbergModel(
    spins={i: 0.5 for i in range(N)},
    exchange={(i, i + 1): 1.0 for i in range(N - 1)},
)
print(f"SU(2)基底次元 (S={total_s}): {model.calc_dim_su2_sector(total_s)}")

params = SolverParameters(num_states=1, output_log=True, num_threads=6)
result = solve_su2(model, total_s=total_s, params=params)
print(f"Energy: {result.energies}")


SU(2)基底次元 (S=0): 16796
Building basis...
Done in 0.0s (6 threads)
Building Hamiltonian...
Done in 0.0s (6 threads)
Diagonalizing Hamiltonian (ground state)...
Done in 0.1s (6 threads)                          
Improving eigenvector...
Done in 0.0s (6 threads)                          
Energy: [-8.682473334398955]


In [3]:
# S=0, 1, 2 のセクターを計算
sectors = [0, 1, 2]
results = {}

for s in sectors:
    dim = model.calc_dim_su2_sector(s)
    print(f"S={s}: dim={dim}")
    params = SolverParameters(num_states=1, output_log=False, num_threads=6)
    results[s] = solve_su2(model, total_s=s, params=params)
    print(f"  E = {results[s].energies[0]:.6f}")


S=0: dim=16796
  E = -8.682473
S=1: dim=125970
  E = -8.502379
S=2: dim=242250
  E = -7.945231


In [4]:
# SU(2)版では SpinOperator を使用
# m_total を指定する必要がある（|S, M⟩ 状態での期待値）

def compute_correlations_su2(model, result, m_total):
    """SU(2)基底での相関関数を計算"""
    Sz_cfs = []
    Sx_cfs = []
    N = model.num_sites
    
    for i in range(1, N):
        Sz_cfs.append(result.correlation_function(
            core.SpinOperator.Sz, 0,
            core.SpinOperator.Sz, i,
            m_total, state_index=0
        ))
        Sx_cfs.append(result.correlation_function(
            core.SpinOperator.Sx, 0,
            core.SpinOperator.Sx, i,
            m_total, state_index=0
        ))
    
    return np.array(Sz_cfs), np.array(Sx_cfs)


In [ ]:
# S=0, M=0 でのスピン相関
Sz_s0, Sx_s0 = compute_correlations_su2(model, results[0], m_total=0)

print("S=0 シングレット状態のスピン相関:")
print("距離 | ⟨Sz Sz⟩      | ⟨Sx Sx⟩      | 差")
print("-" * 55)
for i, (sz, sx) in enumerate(zip(Sz_s0, Sx_s0)):
    print(f"{i+1:4d} | {sz:12.8f} | {sx:12.8f} | {abs(sz-sx):.2e}")
